In [0]:
from pyspark.sql.functions import (
    col, year, month, quarter, dayofmonth, dayofweek,
    date_format, hour, lit, monotonically_increasing_id,
    to_date, concat, datediff, when, round as spark_round
)

In [0]:
storage_account = "stfintechpipeline"
storage_account_key = "YOUR_ACCESS_KEY_HERE"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_account_key
)
CLEANED = f"abfss://cleaned@{storage_account}.dfs.core.windows.net"
CURATED = f"abfss://curated@{storage_account}.dfs.core.windows.net"

In [0]:
df_transactions = spark.read.parquet(f"{CLEANED}/transactions_data/")
df_cards        = spark.read.parquet(f"{CLEANED}/cards_data/")
df_users        = spark.read.parquet(f"{CLEANED}/users_data/")

print(f"transactions : {df_transactions.count():,} rows, {len(df_transactions.columns)} cols")
print(f"cards        : {df_cards.count():,} rows,  {len(df_cards.columns)} cols")
print(f"users        : {df_users.count():,} rows,  {len(df_users.columns)} cols")

transactions : 13,305,915 rows, 13 cols
cards        : 6,146 rows,  10 cols
users        : 2,000 rows,  13 cols


In [0]:

dim_customer = df_users \
    .withColumn("debt_to_income_ratio",
        spark_round(col("total_debt") / when(col("yearly_income") == 0, lit(1))
        .otherwise(col("yearly_income")), 4)) \
    .withColumn("years_to_retirement",
        col("retirement_age") - col("current_age")) \
    .withColumn("age_group",
        when(col("current_age") < 30,  "Under 30")
        .when(col("current_age") < 45, "30-44")
        .when(col("current_age") < 60, "45-59")
        .otherwise("60+")) \
    .withColumn("credit_score_band",
        when(col("credit_score") < 580, "Poor")
        .when(col("credit_score") < 670, "Fair")
        .when(col("credit_score") < 740, "Good")
        .otherwise("Excellent")) \
    .select(
        col("id").alias("client_id"),
        "current_age", "gender", "birth_year", "birth_month",
        "retirement_age", "years_to_retirement", "age_group",
        "yearly_income", "per_capita_income", "total_debt",
        "credit_score", "credit_score_band",
        "debt_to_income_ratio", "num_credit_cards",
        "latitude", "longitude"
    )

print(f"dim_customer : {dim_customer.count():,} rows, {len(dim_customer.columns)} cols")
dim_customer.show(3)


dim_customer : 2,000 rows, 17 cols
+---------+-----------+------+----------+-----------+--------------+-------------------+---------+-------------+-----------------+----------+------------+-----------------+--------------------+----------------+--------+---------+
|client_id|current_age|gender|birth_year|birth_month|retirement_age|years_to_retirement|age_group|yearly_income|per_capita_income|total_debt|credit_score|credit_score_band|debt_to_income_ratio|num_credit_cards|latitude|longitude|
+---------+-----------+------+----------+-----------+--------------+-------------------+---------+-------------+-----------------+----------+------------+-----------------+--------------------+----------------+--------+---------+
|      825|         53|Female|      1966|         11|            66|                 13|    45-59|      59696.0|          29278.0|  127613.0|         787|        Excellent|              2.1377|               5|   34.15|  -117.76|
|     1746|         53|Female|      1966|    

In [0]:

DATASET_END = to_date(lit("2019-10-31"))

dim_card = df_cards \
    .withColumn("acct_open_parsed",
        to_date(concat(lit("01/"), col("acct_open_date")), "dd/MM/yyyy")) \
    .withColumn("card_expiry_parsed",
        to_date(concat(lit("01/"), col("expires")), "dd/MM/yyyy")) \
    .withColumn("account_tenure_days",
        datediff(DATASET_END, col("acct_open_parsed"))) \
    .withColumn("has_expired_card",
        when(col("card_expiry_parsed") < DATASET_END, True).otherwise(False)) \
    .withColumn("days_until_card_expiry",
        datediff(col("card_expiry_parsed"), DATASET_END)) \
    .withColumn("pin_change_recency",
        lit(2019) - col("year_pin_last_changed")) \
    .select(
        col("id").alias("card_id"),
        "client_id", "card_brand", "card_type",
        "has_chip", "num_cards_issued",
        col("spending_limit"),
        "acct_open_date", "account_tenure_days",
        "expires", "has_expired_card", "days_until_card_expiry",
        "year_pin_last_changed", "pin_change_recency"
    ) \
    .drop("acct_open_parsed", "card_expiry_parsed")

print(f"dim_card : {dim_card.count():,} rows, {len(dim_card.columns)} cols")
dim_card.show(3)


dim_card : 6,146 rows, 14 cols
+-------+---------+----------+---------+--------+----------------+--------------+--------------+-------------------+-------+----------------+----------------------+---------------------+------------------+
|card_id|client_id|card_brand|card_type|has_chip|num_cards_issued|spending_limit|acct_open_date|account_tenure_days|expires|has_expired_card|days_until_card_expiry|year_pin_last_changed|pin_change_recency|
+-------+---------+----------+---------+--------+----------------+--------------+--------------+-------------------+-------+----------------+----------------------+---------------------+------------------+
|   4524|      825|      Visa|    Debit|     YES|               2|       24295.0|       09/2002|               6269|12/2022|           false|                  1127|                 2008|                11|
|   2731|      825|      Visa|    Debit|     YES|               2|       21968.0|       04/2014|               2039|12/2020|           false|    

In [0]:

dim_merchant = df_transactions \
    .select(
        "merchant_id", "merchant_city",
        "merchant_state", "mcc", "merchant_category"
    ) \
    .dropDuplicates(["merchant_id"])

print(f"dim_merchant : {dim_merchant.count():,} rows, {len(dim_merchant.columns)} cols")
dim_merchant.show(3)

dim_merchant : 74,831 rows, 5 cols
+-----------+-------------+--------------+----+--------------------+
|merchant_id|merchant_city|merchant_state| mcc|   merchant_category|
+-----------+-------------+--------------+----+--------------------+
|      30928| Indianapolis|            IN|5541|    Service Stations|
|      92741|     Belgrade|            ME|5813|Drinking Places (...|
|      90398|    Rego Park|            NY|4121|Taxicabs and Limo...|
+-----------+-------------+--------------+----+--------------------+
only showing top 3 rows



In [0]:

dim_time = df_transactions \
    .select(to_date(col("transaction_date")).alias("full_date")) \
    .dropDuplicates() \
    .withColumn("date_key",
        (year(col("full_date")) * 10000 +
         month(col("full_date")) * 100 +
         dayofmonth(col("full_date"))).cast("integer")) \
    .withColumn("year",         year(col("full_date"))) \
    .withColumn("quarter",      quarter(col("full_date"))) \
    .withColumn("month",        month(col("full_date"))) \
    .withColumn("month_name",   date_format(col("full_date"), "MMMM")) \
    .withColumn("day_of_month", dayofmonth(col("full_date"))) \
    .withColumn("day_of_week",  date_format(col("full_date"), "EEEE")) \
    .withColumn("is_weekend",
        when(dayofweek(col("full_date")).isin([1, 7]), True)
        .otherwise(False)) \
    .select(
        "date_key", "full_date", "year", "quarter",
        "month", "month_name", "day_of_month",
        "day_of_week", "is_weekend"
    )

print(f"dim_time : {dim_time.count():,} rows, {len(dim_time.columns)} cols")
dim_time.show(3)

dim_time : 3,591 rows, 9 cols
+--------+----------+----+-------+-----+----------+------------+-----------+----------+
|date_key| full_date|year|quarter|month|month_name|day_of_month|day_of_week|is_weekend|
+--------+----------+----+-------+-----+----------+------------+-----------+----------+
|20100109|2010-01-09|2010|      1|    1|   January|           9|   Saturday|      true|
|20100106|2010-01-06|2010|      1|    1|   January|           6|  Wednesday|     false|
|20100110|2010-01-10|2010|      1|    1|   January|          10|     Sunday|      true|
+--------+----------+----+-------+-----+----------+------------+-----------+----------+
only showing top 3 rows



In [0]:

fact_transactions = df_transactions \
    .withColumn("full_date", to_date(col("transaction_date"))) \
    .join(
        dim_time.select("full_date", "date_key"),
        on="full_date",
        how="left"
    ) \
    .withColumn("churn_risk_score", lit(None).cast("double")) \
    .select(
        col("id").alias("transaction_id"),
        "client_id", "card_id", "merchant_id", "date_key",
        "amount_clean", "use_chip", "is_fraud",
        "merchant_category", "errors", "churn_risk_score"
    )

print(f"fact_transactions : {fact_transactions.count():,} rows, {len(fact_transactions.columns)} cols")
fact_transactions.show(3)


fact_transactions : 13,305,915 rows, 11 cols
+--------------+---------+-------+-----------+--------+------------+------------------+--------+--------------------+--------+----------------+
|transaction_id|client_id|card_id|merchant_id|date_key|amount_clean|          use_chip|is_fraud|   merchant_category|  errors|churn_risk_score|
+--------------+---------+-------+-----------+--------+------------+------------------+--------+--------------------+--------+----------------+
|       7475359|     1127|   3869|      39021|20100101|       22.57|Online Transaction|      No|Tolls and Bridge ...|No Error|            NULL|
|       7475354|     1755|   4228|      28666|20100101|       12.01| Swipe Transaction|      No|Package Stores, B...|No Error|            NULL|
|       7475338|      554|   3912|      67570|20100101|        3.51| Swipe Transaction|      No|   Department Stores|No Error|            NULL|
+--------------+---------+-------+-----------+--------+------------+------------------+----

In [0]:

dim_customer.write.mode("overwrite").parquet(f"{CURATED}/dim_customer/")
print("dim_customer written ✓")

dim_card.write.mode("overwrite").parquet(f"{CURATED}/dim_card/")
print("dim_card written ✓")

dim_merchant.write.mode("overwrite").parquet(f"{CURATED}/dim_merchant/")
print("dim_merchant written ✓")

dim_time.write.mode("overwrite").parquet(f"{CURATED}/dim_time/")
print("dim_time written ✓")

fact_transactions.write.mode("overwrite").parquet(f"{CURATED}/fact_transactions/")
print("fact_transactions written ✓")


dim_customer written ✓
dim_card written ✓
dim_merchant written ✓
dim_time written ✓
fact_transactions written ✓


In [0]:

tables = {
    "dim_customer":    f"{CURATED}/dim_customer/",
    "dim_card":        f"{CURATED}/dim_card/",
    "dim_merchant":    f"{CURATED}/dim_merchant/",
    "dim_time":        f"{CURATED}/dim_time/",
    "fact_transactions": f"{CURATED}/fact_transactions/"
}

for name, path in tables.items():
    df = spark.read.parquet(path)
    print(f"{name:25s} — {df.count():>12,} rows | {len(df.columns):>2} cols")


dim_customer              —        2,000 rows | 17 cols
dim_card                  —        6,146 rows | 14 cols
dim_merchant              —       74,831 rows |  5 cols
dim_time                  —        3,591 rows |  9 cols
fact_transactions         —   13,305,915 rows | 11 cols
